# Credit Risk Threshold Lab

A guided walkthrough of binary credit-risk classification, threshold selection, and business cost analysis.

**Positive class:** `default = 1` (borrower defaulted)  
**Negative class:** `default = 0` (borrower repaid)

The central question: which threshold minimizes the expected cost of our mistakes?

## Setup

In [ ]:
import sys
import os

# Add repo root to path so src imports work from the notebook
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import TARGET_COL, RANDOM_STATE, THRESHOLDS
from src.data import load_dataset, basic_cleaning, split_features_target, train_test_split_data, generate_synthetic_dataset
from src.modeling import build_logistic_regression, build_xgboost, train_model, get_predicted_probabilities
from src.evaluation import evaluate_at_threshold, threshold_sweep, choose_best_threshold, compute_roc_auc, compute_pr_auc
from src.plots import (
    plot_class_balance,
    plot_confusion_matrix,
    plot_precision_recall_by_threshold,
    plot_expected_cost_by_threshold,
    plot_roc_curve,
    plot_precision_recall_curve,
)

print("Imports OK")

## 1. Load dataset

Place your CSV in `data/raw/` and update the path below. The target column must be named `default` (0 = repaid, 1 = defaulted).

If no dataset is available, the cell below generates a synthetic imbalanced dataset for demonstration. **Synthetic data is not a real credit dataset.** Results from it are illustrative only.

In [ ]:
RAW_DATA_PATH = "../data/raw/credit_data.csv"
USE_SYNTHETIC = not os.path.exists(RAW_DATA_PATH)

if USE_SYNTHETIC:
    print("No CSV found at", RAW_DATA_PATH)
    print("Generating synthetic demo data (NOT a real credit dataset).")
    df_raw = generate_synthetic_dataset(n_samples=5000, imbalance_ratio=0.12)
else:
    print("Loading real dataset from", RAW_DATA_PATH)
    df_raw = load_dataset(RAW_DATA_PATH)

print(f"Shape: {df_raw.shape}")
df_raw.head()

## 2. Define the positive class

`default = 1` means the borrower defaulted on their loan. This is the event we are trying to predict. Getting it wrong in either direction has a real cost:

- **False negative**: We predicted repayment, but the borrower defaulted. The lender absorbs the loss.
- **False positive**: We predicted default, but the borrower would have repaid. The lender loses a good customer.

In [ ]:
print("Target column:", TARGET_COL)
print("Positive class (default):", df_raw[TARGET_COL].value_counts().to_dict())
default_rate = df_raw[TARGET_COL].mean()
print(f"Default rate: {default_rate:.2%}")

## 3. Check class balance

Class imbalance is the norm in credit risk. A model trained naively will learn to predict the majority class and appear accurate while being useless for catching defaults.

In [ ]:
plot_class_balance(df_raw[TARGET_COL])
plt.show()

print("\nClass counts:")
print(df_raw[TARGET_COL].value_counts())
print(f"\nMinority class (default=1) is {default_rate:.1%} of the data.")

## 4. Clean data and split

In [ ]:
df_clean = basic_cleaning(df_raw)
print(f"Rows after cleaning: {len(df_clean)} (dropped {len(df_raw) - len(df_clean)} rows with nulls or non-numeric columns)")

X, y = split_features_target(df_clean)
X_train, X_test, y_train, y_test = train_test_split_data(X, y)

print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")
print(f"Train default rate: {y_train.mean():.2%}")
print(f"Test default rate:  {y_test.mean():.2%}")

## 5. Train logistic regression

We use `class_weight='balanced'` so the model does not simply learn to predict the majority class. Balanced weights scale the loss for minority class samples proportionally to their underrepresentation.

In [ ]:
lr = build_logistic_regression()
lr = train_model(lr, X_train, y_train)
lr_proba = get_predicted_probabilities(lr, X_test)
print(f"Logistic Regression trained. Predicted probability range: [{lr_proba.min():.3f}, {lr_proba.max():.3f}]")

## 6. Train XGBoost

XGBoost uses gradient boosted trees with shallow depth and conservative learning rate. No scaling needed since tree-based models are scale-invariant.

In [ ]:
xgb = build_xgboost()
xgb = train_model(xgb, X_train, y_train)
xgb_proba = get_predicted_probabilities(xgb, X_test)
print(f"XGBoost trained. Predicted probability range: [{xgb_proba.min():.3f}, {xgb_proba.max():.3f}]")

## 7. Why accuracy is not enough

Before running any real evaluation, let's show what happens if we evaluate naively.

In [ ]:
from sklearn.metrics import accuracy_score

# Naive baseline: always predict non-default
naive_preds = np.zeros(len(y_test), dtype=int)
naive_acc = accuracy_score(y_test, naive_preds)

lr_default_preds = (lr_proba >= 0.5).astype(int)
xgb_default_preds = (xgb_proba >= 0.5).astype(int)

lr_acc = accuracy_score(y_test, lr_default_preds)
xgb_acc = accuracy_score(y_test, xgb_default_preds)

print(f"Naive baseline accuracy (always predict 0): {naive_acc:.3f}")
print(f"Logistic Regression accuracy at 0.50 threshold: {lr_acc:.3f}")
print(f"XGBoost accuracy at 0.50 threshold:            {xgb_acc:.3f}")
print()
print("The naive baseline shows why accuracy alone is misleading.")
print(f"It correctly ignores {naive_acc:.1%} of the time by never predicting a default.")

## 8. Full metrics at default threshold (0.50)

In [ ]:
lr_metrics_50 = evaluate_at_threshold(y_test, lr_proba, 0.50)
xgb_metrics_50 = evaluate_at_threshold(y_test, xgb_proba, 0.50)

lr_roc = compute_roc_auc(y_test, lr_proba)
xgb_roc = compute_roc_auc(y_test, xgb_proba)
lr_pr = compute_pr_auc(y_test, lr_proba)
xgb_pr = compute_pr_auc(y_test, xgb_proba)

summary = pd.DataFrame({
    "Logistic Regression": {**lr_metrics_50, "roc_auc": lr_roc, "pr_auc": lr_pr},
    "XGBoost": {**xgb_metrics_50, "roc_auc": xgb_roc, "pr_auc": xgb_pr},
}).T

display(summary.round(3))

In [ ]:
# Confusion matrices at default threshold
plot_confusion_matrix(y_test, lr_default_preds, title="Logistic Regression -- Threshold 0.50")
plot_confusion_matrix(y_test, xgb_default_preds, title="XGBoost -- Threshold 0.50")

## 9. ROC curve and Precision-Recall curve

In [ ]:
proba_dict = {
    "Logistic Regression": lr_proba,
    "XGBoost": xgb_proba,
}

plot_roc_curve(y_test, proba_dict)
plot_precision_recall_curve(y_test, proba_dict)

**Why PR-AUC matters more than ROC-AUC for imbalanced data.**

ROC-AUC is computed over all thresholds and uses both the true positive rate and the false positive rate. When negatives are very abundant, the false positive rate stays low even when you are making many false positive errors in absolute terms. This makes ROC look flattering.

PR-AUC shows how well the model trades precision for recall as the threshold changes. For imbalanced datasets where the minority class is what we care about catching, PR-AUC is the more honest signal.

## 10. Threshold sweep

We evaluate both models at every threshold from 0.10 to 0.90 and inspect how precision, recall, and F1 change.

In [ ]:
lr_sweep = threshold_sweep(y_test, lr_proba)
xgb_sweep = threshold_sweep(y_test, xgb_proba)

print("Logistic Regression threshold sweep:")
display(lr_sweep[["threshold", "precision", "recall", "f1", "false_positives", "false_negatives"]].round(3))

In [ ]:
print("XGBoost threshold sweep:")
display(xgb_sweep[["threshold", "precision", "recall", "f1", "false_positives", "false_negatives"]].round(3))

In [ ]:
plot_precision_recall_by_threshold(lr_sweep, model_name="Logistic Regression")
plot_precision_recall_by_threshold(xgb_sweep, model_name="XGBoost")

## 11. Business cost assumptions

We now assign dollar costs to each type of error. **These are example assumptions for this exercise, not real business values.** Replace them with estimates from your actual credit risk, finance, or compliance team before making any operational decision.

- **False positive cost: $500** -- We deny or penalize a creditworthy borrower. Lost revenue, potential fairness risk.
- **False negative cost: $5,000** -- We extend credit to a borrower who defaults. The lender absorbs the loss.

The 10:1 ratio reflects that a default loss is typically much larger than the cost of a declined good application. Adjust this to match your actual loan amounts and recovery rates.

In [ ]:
FP_COST = 500
FN_COST = 5000

print(f"False positive cost: ${FP_COST:,}")
print(f"False negative cost: ${FN_COST:,}")
print("\nNote: these are illustrative assumptions, not real business values.")

## 12. Choose cost-minimizing threshold

In [ ]:
lr_best = choose_best_threshold(lr_sweep, FP_COST, FN_COST)
xgb_best = choose_best_threshold(xgb_sweep, FP_COST, FN_COST)

print("Logistic Regression -- Best threshold:")
for k, v in lr_best.items():
    print(f"  {k}: {v}")

print()
print("XGBoost -- Best threshold:")
for k, v in xgb_best.items():
    print(f"  {k}: {v}")

In [ ]:
plot_expected_cost_by_threshold(lr_sweep, FP_COST, FN_COST, model_name="Logistic Regression")
plot_expected_cost_by_threshold(xgb_sweep, FP_COST, FN_COST, model_name="XGBoost")

## 13. Compare models at their optimal thresholds

In [ ]:
lr_opt_threshold = lr_best["threshold"]
xgb_opt_threshold = xgb_best["threshold"]

lr_opt_metrics = evaluate_at_threshold(y_test, lr_proba, lr_opt_threshold)
xgb_opt_metrics = evaluate_at_threshold(y_test, xgb_proba, xgb_opt_threshold)

comparison = pd.DataFrame({
    f"LR (t={lr_opt_threshold})": {**lr_opt_metrics, "expected_cost": lr_best["expected_cost"]},
    f"XGB (t={xgb_opt_threshold})": {**xgb_opt_metrics, "expected_cost": xgb_best["expected_cost"]},
}).T

display(comparison.round(3))

In [ ]:
# Confusion matrices at optimal thresholds
lr_opt_preds = (lr_proba >= lr_opt_threshold).astype(int)
xgb_opt_preds = (xgb_proba >= xgb_opt_threshold).astype(int)

plot_confusion_matrix(y_test, lr_opt_preds, title=f"Logistic Regression -- Threshold {lr_opt_threshold}")
plot_confusion_matrix(y_test, xgb_opt_preds, title=f"XGBoost -- Threshold {xgb_opt_threshold}")

## 14. SHAP feature importance (XGBoost)

SHAP values show how much each feature pushed the model's prediction up or down for each individual borrower. This is useful for explainability and for checking whether the model is relying on features that would raise fairness or compliance concerns.

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, plot_type="bar", show=True)
except ImportError:
    print("shap not installed. Run: pip install shap")
except Exception as e:
    print(f"SHAP plot failed: {e}")

## 15. Business stakeholder summary

After running the notebook, this section summarizes the results in plain language.

Fill in the values below from your actual run, then copy this section to `reports/stakeholder_summary.md`.

In [ ]:
# Determine the winning model by expected cost
if lr_best["expected_cost"] <= xgb_best["expected_cost"]:
    winner_name = "Logistic Regression"
    winner_best = lr_best
    winner_roc = lr_roc
    winner_pr = lr_pr
else:
    winner_name = "XGBoost"
    winner_best = xgb_best
    winner_roc = xgb_roc
    winner_pr = xgb_pr

print("="*60)
print("STAKEHOLDER SUMMARY (auto-generated from notebook run)")
print("="*60)
print(f"Dataset: {'Synthetic demo' if USE_SYNTHETIC else RAW_DATA_PATH}")
print(f"Default rate: {default_rate:.2%}")
print()
print(f"Recommended model:   {winner_name}")
print(f"Recommended threshold: {winner_best['threshold']}")
print(f"Precision:             {winner_best['precision']:.3f}")
print(f"Recall:                {winner_best['recall']:.3f}")
print(f"F1:                    {winner_best['f1']:.3f}")
print(f"ROC-AUC:               {winner_roc:.3f}")
print(f"PR-AUC:                {winner_pr:.3f}")
print(f"False positives (test): {int(winner_best['false_positives'])}")
print(f"False negatives (test): {int(winner_best['false_negatives'])}")
print(f"Expected cost (test):  ${winner_best['expected_cost']:,.0f}")
print()
print(f"Cost assumptions: FP=${FP_COST:,} | FN=${FN_COST:,} (illustrative only)")
print("="*60)

## 16. Final reflections

**What this exercise demonstrated:**

1. Accuracy is a misleading metric when classes are imbalanced. A naive model that predicts no defaults can appear highly accurate while catching zero of the events we care about.

2. The threshold is not a hyperparameter of the model. It is a business decision. Moving from 0.50 to a lower threshold increases recall (catch more defaults) at the cost of precision (more false alarms). Moving higher does the opposite.

3. The precision-recall curve is the right place to look when the minority class is what you care about. ROC can look flattering on imbalanced data even when the model is making many false positive errors.

4. Attaching a cost to each type of error converts a modeling problem into a business optimization. The cost-minimizing threshold is a defensible recommendation that ties the model to real consequences.

5. The stakeholder summary matters as much as the model. Explaining false positives and false negatives in plain language is part of the job.

**Limitations of this exercise:**

- The cost assumptions are illustrative. Real cost analysis requires input from finance and credit risk teams.
- A single train-test split can be noisy. Cross-validation with threshold selection on each fold gives more stable estimates.
- No fairness analysis was performed. In production, false positive rates should be checked across demographic groups.
- This did not include resampling techniques (SMOTE, undersampling) which may improve recall on highly imbalanced datasets.
- The synthetic dataset (if used) has no meaningful feature semantics. Results from it should not be interpreted as credit domain findings.